# 03 - Preprocesamiento NLP y Vectorización (TF-IDF)

En este notebook vamos a convertir nuestros tickets de texto a formato numérico. Los algoritmos de Machine Learning no entienden palabras, solo números, por lo que necesitamos traducir el lenguaje natural a una representación matemática.

Para lograrlo, aplicaremos dos transformaciones:
1. **Lematización y limpieza (spaCy):** Reduciremos las palabras a su raíz (por ejemplo, "running" pasará a ser "run") y eliminaremos palabras comunes que no aportan valor (stop words como "the", "is", "at") así como los signos de puntuación.
2. **Vectorización (TF-IDF):** Usaremos `TfidfVectorizer` para crear una matriz donde cada columna será una palabra del vocabulario y cada fila será un ticket. El valor TF-IDF le da más peso matemático a las palabras raras y específicas (como "router" o "password") frente a las comunes.

Finalmente, guardaremos estas matrices numéricas en formato `.npz` (Sparse Matrix) para poder cargarlas rápidamente en los cuadernos de modelado.

In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import time

# 1. Cargamos los textos puros que guardamos en el notebook anterior
X_train = pd.read_csv("../data/features/en_X_train_text.csv")['texto_completo']
X_test = pd.read_csv("../data/features/en_X_test_text.csv")['texto_completo']

# 2. Cargamos el modelo de lenguaje de spaCy en inglés
# Deshabilitamos partes avanzadas ('ner', 'parser') porque solo queremos limpiar y lematizar, así va más rápido.
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

# Función para limpiar el texto línea a línea
def limpiar_texto(texto):
    # Pasamos a minúsculas y se lo damos a spaCy
    doc = nlp(str(texto).lower())
    # Guardamos el lema (raíz) si la palabra no es stopword, ni puntuación, ni espacio en blanco
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    return " ".join(tokens)

print("Iniciando limpieza NLP con spaCy (esto puede tardar un par de minutos)...")
start_time = time.time()

# Aplicamos la función de limpieza a nuestros datos de Train y Test
X_train_clean = X_train.apply(limpiar_texto)
X_test_clean = X_test.apply(limpiar_texto)

print(f"Limpieza NLP terminada en {round((time.time() - start_time)/60, 2)} minutos.")

# 3. Vectorización TF-IDF
print("\nIniciando vectorización TF-IDF limitando a 10.000 palabras (dimensiones)...")
vectorizer = TfidfVectorizer(max_features=10000)

# MUY IMPORTANTE: Hacemos fit_transform solo en Train para que aprenda el vocabulario
X_train_tfidf = vectorizer.fit_transform(X_train_clean)

# En el Test solo hacemos transform (no fit) para no hacer trampa aprendiendo palabras del examen
X_test_tfidf = vectorizer.transform(X_test_clean)

print("Forma de la matriz de Train:", X_train_tfidf.shape)
print("Forma de la matriz de Test:", X_test_tfidf.shape)

# 4. Guardado físico de las matrices
# Guardamos en formato Sparse (.npz) porque al haber miles de ceros, este formato ahorra muchísima memoria RAM
sparse.save_npz("../data/features/en_X_train_tfidf.npz", X_train_tfidf)
sparse.save_npz("../data/features/en_X_test_tfidf.npz", X_test_tfidf)

print("\nMatrices numéricas guardadas correctamente en data/features/")

Iniciando limpieza NLP con spaCy (esto puede tardar un par de minutos)...
Limpieza NLP terminada en 2.09 minutos.

Iniciando vectorización TF-IDF limitando a 10.000 palabras (dimensiones)...
Forma de la matriz de Train: (18493, 4717)
Forma de la matriz de Test: (4624, 4717)

Matrices numéricas guardadas correctamente en data/features/
